In [106]:
from cs336_basics import model as mt_model
from cs336_basics import BPETokenizer
import torch
from pathlib import Path
import time
import logging
import os

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

current_dir = Path(os.getcwd())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [107]:
tiny_vocab_path = str((current_dir / "../vocab_TinyStories.json").resolve())
tiny_merges_path = str((current_dir / "../merges_TinyStories.json").resolve())
tiny_tokenizer = BPETokenizer.from_pretrained(
    vocab_path=tiny_vocab_path,
    merges_path=tiny_merges_path,
    )
print(f"Tokenizer loaded from {tiny_vocab_path} and {tiny_merges_path}")
print(f"Tokenizer vocabs from 0 to 100: {[tiny_tokenizer.vocab[i] for i in range(100)]}")
print(f"Tokenizer <|endoftext|> id: {tiny_tokenizer.token2id[b"<|endoftext|>"]}")

Tokenizer loaded from /data/satori_hdd1/mutyuu/workspace/CS336/assignment1/experiments/vocab_TinyStories.json and /data/satori_hdd1/mutyuu/workspace/CS336/assignment1/experiments/merges_TinyStories.json
Tokenizer vocabs from 0 to 100: [b'<|endoftext|>', b'\x00', b'\x01', b'\x02', b'\x03', b'\x04', b'\x05', b'\x06', b'\x07', b'\x08', b'\t', b'\n', b'\x0b', b'\x0c', b'\r', b'\x0e', b'\x0f', b'\x10', b'\x11', b'\x12', b'\x13', b'\x14', b'\x15', b'\x16', b'\x17', b'\x18', b'\x19', b'\x1a', b'\x1b', b'\x1c', b'\x1d', b'\x1e', b'\x1f', b' ', b'!', b'"', b'#', b'$', b'%', b'&', b"'", b'(', b')', b'*', b'+', b',', b'-', b'.', b'/', b'0', b'1', b'2', b'3', b'4', b'5', b'6', b'7', b'8', b'9', b':', b';', b'<', b'=', b'>', b'?', b'@', b'A', b'B', b'C', b'D', b'E', b'F', b'G', b'H', b'I', b'J', b'K', b'L', b'M', b'N', b'O', b'P', b'Q', b'R', b'S', b'T', b'U', b'V', b'W', b'X', b'Y', b'Z', b'[', b'\\', b']', b'^', b'_', b'`', b'a', b'b']
Tokenizer <|endoftext|> id: 0


In [108]:
model_config = {
    "d_model": 512,
    "num_heads": 16,
    "d_ff": 1344,
    "num_layers": 4,
    "vocab_size": 10000,
    "max_seq_len": 256,
    "theta": 10000.0,
}
tiny_model = mt_model.Transformer(
    model_config["d_model"],
    model_config["num_heads"],
    model_config["d_ff"],
    model_config["num_layers"],
    model_config["vocab_size"],
    model_config["max_seq_len"],
    model_config["theta"],
)
tiny_model.to(device)
mt_model.load_checkpoint(
    src="checkpoints/checkpoint_iter_5000.pth",
    model=tiny_model,
    optimizer=None,
)

5000

In [ ]:
prompt = "I hate Mondays."
prompt_ids = tiny_tokenizer.encode(prompt)
prompt_tensor = torch.tensor(prompt_ids, dtype=torch.long).to(device)
generated_ids = tiny_model.generate(
    input_ids=prompt_tensor,
    max_length=256,
    temperature=1.0,
    top_p=0.9,
)
generated_text = tiny_tokenizer.decode(generated_ids.tolist())
print(f"Prompt: {prompt}")
print(f"Prompt IDs: {prompt_ids}")
print(f"Generated IDs: {generated_ids.tolist()}")
print(f"Generated text: {generated_text}")

Prompt: I hate Mondays.
Prompt IDs: [74, 4261, 379, 2856, 5813, 47]
Generated IDs: [74, 4261, 379, 2856, 5813, 47, 1609, 375, 338, 375, 476, 392, 541, 328, 5876, 47, 11, 35, 1308, 431, 309, 391, 2506, 476, 336, 386, 1468, 47, 11, 81, 105, 314, 806, 492, 1648, 45, 408, 286, 548, 266, 951, 267, 324, 45, 317, 74, 923, 365, 2506, 34, 338, 516, 728, 954, 397, 11, 410, 386, 562, 267, 324, 45, 317, 1146, 946, 1378, 271, 3451, 267, 1425, 263, 1569, 45, 448, 737, 448, 349, 483, 864, 267, 797, 397, 11, 61, 125, 353, 112, 791, 3273, 125, 63, 11, 383, 327, 45, 259, 390, 465, 402, 514, 283, 584, 313, 263, 1247, 47, 514, 528, 266, 716, 267, 736, 623, 47, 316, 382, 259, 346, 497, 328, 661, 2760, 47, 514, 407, 266, 325, 328, 263, 2760, 47, 316, 945, 267, 945, 45, 408, 286, 466, 365, 1470, 263, 2760, 47, 11, 66, 797, 477, 402, 406, 382, 514, 313, 263, 497, 47, 339, 324, 45, 317, 1096, 45, 514, 34, 1195, 349, 369, 266, 325, 328, 519, 476, 514, 4654, 988, 47, 316, 945, 267, 473, 328, 263, 2760, 45, 1222,